In [ ]:
#!pip uninstall -y docx
#!pip install -U python-docx

In [ ]:
import re
import os
from docx import Document
import pandas as pd
# === FILE PATHS ===
INPUT_BASE_DIR = "/SURVEY TRANSCRIPT"
INPUT_DOCX_FILES = [
    os.path.join(INPUT_BASE_DIR, f"Study_Transcript{i}.docx")
    for i in range(1, 14)
]
OUTPUT_TEMPLATE = "FGD{}_transcript_turns.csv"
# Optional: manually rescue lines that do not match regex
MANUAL_PREFIX_TO_SPEAKER = {}
# Allow bullets and alternate colons; loosen speaker chars to catch more lines
SPEAKER_LINE_RE = re.compile(r'^(?:[-•]\s*)?([A-Za-z][A-Za-z\s()#0-9/\.-]*?)[:：]\s*(.+)$')
def infer_role(speaker_name: str, speaker_raw: str) -> str:
    s_lower = speaker_name.lower()
    raw_lower = speaker_raw.lower()
    if "moderator" in s_lower or speaker_name in {""}:
        return "Moderator"
    if "observer" in raw_lower or speaker_name in {""}:
        return "Observer"
    return "Respondent"
def clean_utterance(text: str) -> str:
    if not isinstance(text, str):
        return ""
    text = text.strip()
    text = re.sub(r"\s+", " ", text)
    return text
def extract_turns_from_docx(path: str, session_id: str = None):
    doc = Document(path)
    rows = []
    last_turn = None
    stats = {
        "paragraphs": len(doc.paragraphs),
        "lines_total": 0,
        "lines_matched": 0,
        "lines_skipped_empty": 0,
        "lines_skipped_header": 0,
        "lines_skipped_nonmatch": 0,
        "unmatched_lines": [],
        "unmatched_samples": [],
    }
    turn_id = 0
    skip_speakers = {
        "Date", "Time", "Venue",
        "Respondents, n=7",
        "Brand Study 2025",
        "Transcript for FGD 1",
    }
    for p in doc.paragraphs:
        text = p.text
        if not text:
            stats["lines_skipped_empty"] += 1
            continue
        for line in text.split("\n"):
            line = line.strip()
            if not line:
                stats["lines_skipped_empty"] += 1
                continue
            stats["lines_total"] += 1
            m = SPEAKER_LINE_RE.match(line)
            manual_speaker = None
            manual_prefix_len = 0
            if not m and MANUAL_PREFIX_TO_SPEAKER:
                for prefix, speaker in MANUAL_PREFIX_TO_SPEAKER.items():
                    if line.startswith(prefix):
                        manual_speaker = speaker
                        manual_prefix_len = len(prefix)
                        break
            if m:
                speaker_raw = m.group(1).strip()
                utterance = m.group(2).strip()
            elif manual_speaker:
                speaker_raw = manual_speaker
                utterance = line[manual_prefix_len:].lstrip(":：-• ").strip() or line
            elif last_turn is not None:
                continuation = clean_utterance(line)
                if continuation:
                    last_turn["utterance"] += " " + continuation
                    stats["lines_matched"] += 1
                else:
                    stats["lines_skipped_empty"] += 1
                continue
            else:
                stats["lines_skipped_nonmatch"] += 1
                stats["unmatched_lines"].append(line)
                if len(stats["unmatched_samples"]) < 3:
                    stats["unmatched_samples"].append(line)
                continue
            if any(speaker_raw.startswith(x) for x in skip_speakers):
                stats["lines_skipped_header"] += 1
                continue
            speaker_name = re.sub(r"\s*\(.*?\)", "", speaker_raw).strip()
            speaker_role = infer_role(speaker_name, speaker_raw)
            utterance_clean = clean_utterance(utterance)
            if not utterance_clean:
                stats["lines_skipped_empty"] += 1
                continue
            turn_id += 1
            turn = {
                "session_id": session_id,
                "turn_id": turn_id,
                "speaker_raw": speaker_raw,
                "speaker_name": speaker_name,
                "speaker_role": speaker_role,
                "utterance": utterance_clean,
            }
            rows.append(turn)
            last_turn = turn
            stats["lines_matched"] += 1
    df = pd.DataFrame(rows)
    stats["turns"] = len(rows)
    return df, stats
# === RUN THE EXTRACTION FOR MULTIPLE FILES ===
all_dfs = []
for idx, path in enumerate(INPUT_DOCX_FILES, start=1):
    if not os.path.exists(path):
        print(f"[FGD {idx}] Missing file, skipping: {path}")
        continue
    session_id = os.path.splitext(os.path.basename(path))[0]
    df, stats = extract_turns_from_docx(path, session_id=session_id)
    out_csv = OUTPUT_TEMPLATE.format(idx)
    df.to_csv(out_csv, index=False, encoding="utf-8-sig")
    print(
        f"[FGD {idx}] {out_csv}: turns={len(df)} matched={stats['lines_matched']} "
        f"nonmatch={stats['lines_skipped_nonmatch']} empty={stats['lines_skipped_empty']}"
    )
    if stats["unmatched_lines"]:
        sample = stats["unmatched_samples"] or stats["unmatched_lines"][:3]
        print(f"  unmatched sample: {sample}")
    all_dfs.append(df)
if all_dfs:
    combined = pd.concat(all_dfs, ignore_index=True)
    combined.to_csv("FGD_all_transcript_turns.csv", index=False, encoding="utf-8-sig")
    print("\nCombined export saved: FGD_all_transcript_turns.csv")